[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [32]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        pass  # Initialize projections

        self.head_size = d_model // num_heads
        self.num_q_heads = num_heads
        self.num_kv_heads = num_kv_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.head_size)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.head_size)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        pass  # Self-attention with grouped KV
        Q = self.W_q(x)      #(batch_size, seq, d_model) = (batch_size, seq, num_q_heads * head_size)
        K = self.W_k(x)      #(batch_size, seq, num_kv_heads * head_size)
        V = self.W_v(x)      #(batch_size, seq, num_kv_heads * head_size)

        K_repeated = torch.repeat_interleave(K, self.num_q_heads // self.num_kv_heads, dim=-1)   #(batch_size, seq, num_q_heads * head_size)
        V_repeated = torch.repeat_interleave(V, self.num_q_heads // self.num_kv_heads, dim=-1)   #(batch_size, seq, num_q_heads * head_size)


        attention_scores = []

        for i in range(self.num_q_heads):
          Q_i = Q[:, :, i*self.head_size : (i+1)*self.head_size]           #(batch_size, seq, head_size)
          K_i = K_repeated[:, :, i*self.head_size : (i+1)*self.head_size]  #(batch_size, seq, head_size)
          V_i = V_repeated[:, :, i*self.head_size : (i+1)*self.head_size]  #(batch_size, seq, head_size)

          K_i_T = K_i.transpose(1, 2)                                      #(batch_size, head_size, seq)
          QK_i = (Q_i @ K_i_T) / math.sqrt(self.head_size)                 #(batch_size, seq, seq)

          mask = torch.ones_like(QK_i)
          mask = torch.where(torch.tril(mask) == 1, 0, float('-inf'))
          QK_i += mask

          softmax_scores_i = torch.softmax(QK_i, dim=-1)        #(batch_size, seq, seq)
          attention_scores_i = softmax_scores_i @ V_i           #(batch_size, seq, head_size)
          attention_scores.append(attention_scores_i)

        attention_scores = torch.cat(attention_scores, dim=-1)  #(batch_size, seq, d_model)
        attention_scores = self.W_o(attention_scores)           #(batch_size, seq, d_model)   #???

        return attention_scores

In [33]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [34]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (7.4ms)
  ✅ [2/5] nn.Linear with correct shapes (0.7ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (3.1ms)
  ✅ [4/5] KV heads are shared correctly (2.5ms)
  ✅ [5/5] Gradient flow (4.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (18.0ms total)
  Progress saved. Run status() to see your dashboard.

